# 10. Shared alert budget across five coins

As a systems-engineering question of resource pooling, I ask which catches more crashes, and earlier: separate analyst alert-handling capacity for each coin (240 per year and 20 per month per coin), or one total that five coins share (1,200 per year and 100 per month). Crypto crashes often occur in several coins at the same time, so pooling may help (spare capacity gets used) or hurt (capacity runs out everywhere at once).

- For BTC I use the spot cache and the notebook 05 model unchanged; for ETH, SOL, DOGE and AVAX I use Binance spot one-minute bars from 2020-01 or the coin's listing, whichever is later.
- BTC keeps the existing 4% event rule. The other coins are more volatile, so 4% would produce too many events; I fix their drop size in advance at `4% × (the coin's 2021 median daily realized volatility / the same value for BTC)`. The other rules (240-minute peak, 5-minute confirmation, 24-hour separation) stay unchanged.
- I compare three policies (V+F logistic, which I train on 2021, calibrate on 2022 and evaluate on 2023 to 2026-04):
  - S0 separate: 20 per month and 240 per year for each coin, with coin-specific 2022 thresholds; for BTC this is the notebook 05 policy.
  - S1 pooled: a joint 100 per month and 1,200 per year for the five coins. I convert scores to percentiles of each coin's 2022 distribution so that they are comparable, and I calibrate one common percentile threshold on 2022.
  - S1b pooled + per-coin cap: S1 plus a cap of 40 alerts per coin per month, so that a single coin cannot take the whole budget.
- I write the results to `btc_lead_cost_results_77months_spot/multicoin_shared_budget_v1/`.
- This analysis is exploratory.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df


OUT = SPOT_BASE / 'multicoin_shared_budget_v1'
CACHE = OUT / 'cache'; CACHE.mkdir(parents=True, exist_ok=True)
ZIPS = Path('/content/alt_spot_zips') if Path('/content').exists() else OUT / 'zips'
ALTS = ['ETHUSDT', 'SOLUSDT', 'DOGEUSDT', 'AVAXUSDT']
COINS = ['BTCUSDT'] + ALTS
FIRST, LAST = '2020-01', '2026-04'
POOL_B = B * len(COINS); POOL_CAP = MONTH_CAP * len(COINS); COIN_CAP_S1B = 2 * MONTH_CAP
SPEC = {'study': 'shared alert budget across coins, exploratory', 'coins': COINS, 'months': [FIRST, LAST],
        'event_rule': '4% x sigma_coin/sigma_BTC (median 2021 daily realized vol of 1m log returns); 240-min peak; 5 closes; 24h refractory',
        'model': 'V+F logistic per coin, 2021 train (BTC identical to notebook 5)',
        'S0': 'separate calendar 20/month, 240/yr per coin; notebook-5 calibration rule per coin on 2022',
        'S1': f'pooled calendar {POOL_CAP}/month, {POOL_B}/yr; per-coin 240-min cooldown; score = coin-specific 2022 percentile; one percentile threshold calibrated on 2022 (most alerts reachable, then strictest)',
        'S1b': f'S1 plus at most {COIN_CAP_S1B} alerts per coin per month',
        'evaluation': f'{EVAL_START} to {EVAL_END} exclusive', 'primary_endpoint': 'pre-1%-onset hits summed over coins',
        'status': 'exploratory'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True) + '\n').encode()
(OUT / 'multicoin_spec.json').write_bytes(_raw)
print('Output:', OUT)

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/multicoin_shared_budget_v1


## 1. Altcoin spot one-minute bars

In [2]:
manifests = []
for sym in ALTS:
    pq = CACHE / f'{sym}_1m.parquet'
    if pq.is_file():
        print(sym, 'cache reused'); continue
    man = get_archives('spot', sym, month_list(FIRST, LAST), ZIPS / sym, allow_missing=True)
    man.to_csv(OUT / f'{sym}_download_manifest.csv', index=False)
    raw = load_archives(man, ZIPS / sym)
    reg = regularize(raw, end=pd.Timestamp('2026-04-30 23:59', tz='UTC'))
    reg = reg.loc[:pd.Timestamp('2026-04-30 23:59', tz='UTC')]
    feats = compute_spot_features(reg, with_hs=False)
    out = pd.concat([reg[['close', 'volume', 'taker_buy_vol']], feats], axis=1)
    out.to_parquet(pq)
    print(sym, out.index[0], '->', out.index[-1], f'{len(out):,} minutes; missing long-gap minutes {int(out.close.isna().sum()):,}')
    del raw, reg, feats, out

ETHUSDT spot {'downloaded': 76}
ETHUSDT 2020-01-01 00:00:00+00:00 -> 2026-04-30 23:59:00+00:00 3,329,280 minutes; missing long-gap minutes 2,326
SOLUSDT spot {'downloaded': 69, 'not_listed': 7}
SOLUSDT 2020-08-11 06:00:00+00:00 -> 2026-04-30 23:59:00+00:00 3,007,800 minutes; missing long-gap minutes 1,424
DOGEUSDT spot {'downloaded': 76}
DOGEUSDT 2020-01-01 00:00:00+00:00 -> 2026-04-30 23:59:00+00:00 3,329,280 minutes; missing long-gap minutes 2,326
AVAXUSDT spot {'downloaded': 68, 'not_listed': 8}
AVAXUSDT 2020-09-22 06:30:00+00:00 -> 2026-04-30 23:59:00+00:00 2,947,290 minutes; missing long-gap minutes 1,424


## 2. Events, models and scores per coin

In [ ]:
px, sg = load_btc_spot()
E_btc = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')


def daily_sigma_2021(index, close):
    r = pd.Series(np.log(close), index=index).diff()
    d = r.loc['2021'].groupby(r.loc['2021'].index.floor('D')).agg(['std', 'count'])
    return float((d.loc[d['count'] >= 1000, 'std'] * np.sqrt(1440)).median())


coin = {}
sig_btc = daily_sigma_2021(px.index, px.close.to_numpy(float))
for sym in COINS:
    if sym == 'BTCUSDT':
        idx = px.index; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
        v = sg.z_vol.to_numpy(float); f = sg.tsr.to_numpy(float); h = sg.z_hs.to_numpy(float)
        valid = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v) & np.isfinite(f) & np.isfinite(h) & (f >= 0) & (f <= 1))
        E = E_btc; decline = 0.04
    else:
        d = pd.read_parquet(CACHE / f'{sym}_1m.parquet')
        d.index = d.index.tz_convert('UTC')
        idx = d.index; close = d.close.to_numpy(float); volume = d.volume.to_numpy(float)
        v = d.z_vol.to_numpy(float); f = d.tsr.to_numpy(float)
        valid = np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v) & np.isfinite(f) & (f >= 0) & (f <= 1)
        decline = 0.04 * daily_sigma_2021(idx, close) / sig_btc
        E = price_only_catalog(close, volume, idx, decline=decline)
        E.to_csv(OUT / f'{sym}_price_only_event_catalog.csv', index=False)
    T = idx.asi8
    y = label_from_catalog(T, E)
    tr = train_mask(T, valid)
    if not (0 < y[tr].sum() < tr.sum()):
        raise RuntimeError(f'{sym}: training labels have one class')
    m = new_logit().fit(np.column_stack([v, f])[tr], y[tr])
    S = np.full(len(T), np.nan); S[valid] = m.predict_proba(np.column_stack([v, f])[valid])[:, 1]
    cal = valid & in_period(T, '2022-01-01', '2023-01-01'); ev = valid & in_period(T, EVAL_START, EVAL_END)
    ref22 = np.sort(S[cal])
    coin[sym] = dict(T_CAL=T[cal], S_CAL=S[cal], T_EV=T[ev], S_EV=S[ev], ref22=ref22, decline=decline,
                     EVENTS=events_between(E, EVAL_START, EVAL_END), n_events_all=len(E),
                     events_by_year=E.groupby(E.breach_utc.dt.year).size().to_dict())
    del S, y
    print(f'{sym}: decline {decline:.4f} | events all {len(E)} | eval events {len(coin[sym]["EVENTS"])} | by year {coin[sym]["events_by_year"]}')
assert len(coin['BTCUSDT']['EVENTS']) == REF['n_primary_events']

## 3. S0: separate budget per coin

In [ ]:
A0 = {}; rows = []
c_ = coin['BTCUSDT']; print('fast calendar identical to the reference on', check_fast_calendar(c_['T_CAL'], c_['S_CAL']), 'checks')
for sym, c in coin.items():
    th, ncal, _ = calibrate_calendar(c['T_CAL'], c['S_CAL'])
    c['th'] = th
    A0[sym] = emit_calendar_fast(c['T_EV'], c['S_EV'] >= th)
    st = statistics(c['EVENTS'], A0[sym])
    rows.append(dict(policy='S0_separate', coin=sym, threshold_2022=th, cal_alerts_2022=ncal, events=len(c['EVENTS']),
                     alarms=st['alarms'], caught=st['caught'], before_1pct=st['before_1pct']))
    if sym == 'BTCUSDT':
        if abs(th - TH_CAL) > 1e-12 or [st['alarms'], st['caught'], st['before_1pct']] != REF['first_eligible']:
            raise RuntimeError(f'BTC S0 differs from notebook 5: {th} {st}')
        print('PASS: BTC S0 reproduces notebook 5', REF['first_eligible'])
s0 = pd.DataFrame(rows); display(s0)

fast calendar identical to the reference on 14 checks
PASS: BTC S0 reproduces notebook 5 [800, 39, 22]


,policy,coin,threshold_2022,cal_alerts_2022,events,alarms,caught,before_1pct
0,S0_separate,BTCUSDT,0.669899,240,83,800,39,22
1,S0_separate,ETHUSDT,0.676319,240,94,800,35,9
2,S0_separate,SOLUSDT,0.710229,240,68,800,32,12
3,S0_separate,DOGEUSDT,0.715425,240,112,792,42,16
4,S0_separate,AVAXUSDT,0.705006,240,40,800,16,2


## 4. S1 / S1b: shared budget

In [ ]:
@njit(cache=False)
def _pooled_core(t, cid, years, months, B_total, cap_month, cooldown_ns, ncoins, coin_cap):
    out = np.empty(len(t), np.int64); k = 0
    last = np.full(ncoins, -(2**62), np.int64); ccount = np.zeros(ncoins, np.int64)
    mcount = 0; ycount = 0; pm = -1; py = -1
    for i in range(len(t)):
        yr = years[i]; mo = months[i]; c = cid[i]; ti = t[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
            for j in range(ncoins):
                ccount[j] = 0
        if ycount >= B_total or mcount >= cap_month:
            continue
        if coin_cap > 0 and ccount[c] >= coin_cap:
            continue
        if ti - last[c] < cooldown_ns:
            continue
        out[k] = i; k += 1; last[c] = ti; ycount += 1; mcount += 1; ccount[c] += 1
    return out[:k].copy()


def pooled_stream(part):
    """Merge all coins' minutes of one period into one time-ordered stream (ties: higher percentile first)."""
    ts_, cid, u = [], [], []
    for j, sym in enumerate(COINS):
        c = coin[sym]
        t = c[f'T_{part}']; s = c[f'S_{part}']
        ts_.append(t); cid.append(np.full(len(t), j, np.int64))
        u.append(np.searchsorted(c['ref22'], s, 'right') / len(c['ref22']))
    t = np.concatenate(ts_); cid = np.concatenate(cid); u = np.concatenate(u)
    order = np.lexsort((-u, t))
    t, cid, u = t[order], cid[order], u[order]
    yrs, mons = cal_fields(t)
    return t, cid, u, yrs, mons


def run_pool(stream, thr, coin_cap=0):
    t, cid, u, yrs, mons = stream
    sel = u >= thr
    k = _pooled_core(t[sel], cid[sel], yrs[sel], mons[sel], POOL_B, POOL_CAP, np.int64(COOLDOWN * NS), len(COINS), coin_cap)
    tt = t[sel][k]; cc = cid[sel][k]
    return {sym: tt[cc == j] for j, sym in enumerate(COINS)}


cal_stream = pooled_stream('CAL'); ev_stream = pooled_stream('EV')
ugrid = np.unique(np.r_[np.quantile(cal_stream[2], np.r_[np.linspace(.90, .999, 45), .9995, .9999]), 1.0])
results = {}
for name, cc in [('S1_pooled', 0), ('S1b_pooled_coincap', COIN_CAP_S1B)]:
    grid = []
    for thr in ugrid:
        n = sum(len(a) for a in run_pool(cal_stream, thr, cc).values())
        grid.append((float(thr), n, int((cal_stream[2] >= thr).sum())))
    g = pd.DataFrame(grid, columns=['u_threshold', 'cal_alerts', 'raw_candidates'])
    thr = float(g[g.cal_alerts == g.cal_alerts.max()].sort_values('raw_candidates').iloc[0].u_threshold)
    g.to_csv(OUT / f'{name}_threshold_search_2022.csv', index=False)
    results[name] = (thr, run_pool(ev_stream, thr, cc))
    print(name, 'percentile threshold', round(thr, 5), '2022 alerts', int(g.cal_alerts.max()))

S1_pooled percentile threshold 0.95625 2022 alerts 1200
S1b_pooled_coincap percentile threshold 0.95625 2022 alerts 1200


## 5. Comparison

In [ ]:
rows = [r for r in s0.to_dict('records')]
for name, (thr, al) in results.items():
    for sym in COINS:
        st = statistics(coin[sym]['EVENTS'], al[sym])
        rows.append(dict(policy=name, coin=sym, threshold_2022=thr, events=len(coin[sym]['EVENTS']),
                         alarms=st['alarms'], caught=st['caught'], before_1pct=st['before_1pct']))
tab = pd.DataFrame(rows); tab.to_csv(OUT / 'multicoin_policy_by_coin.csv', index=False)
tot = tab.groupby('policy')[['events', 'alarms', 'caught', 'before_1pct']].sum()
tot.to_csv(OUT / 'multicoin_policy_totals.csv')
display(tab.pivot(index='coin', columns='policy', values='before_1pct')); display(tot)

pair_rows = []
for name, (thr, al) in results.items():
    a_only = b_only = a_only_h = b_only_h = 0
    for sym in COINS:
        p = paired(coin[sym]['EVENTS'], A0[sym], al[sym])
        a_only += p['base_only_pre1pct']; b_only += p['new_only_pre1pct']
        a_only_h += p['base_only_caught']; b_only_h += p['new_only_caught']
    pair_rows.append(dict(policy=name, separate_only_pre1pct=a_only, pooled_only_pre1pct=b_only,
                          p_pre1pct_two_sided=mcnemar_two_sided(a_only, b_only),
                          separate_only_caught=a_only_h, pooled_only_caught=b_only_h,
                          p_caught_two_sided=mcnemar_two_sided(a_only_h, b_only_h)))
pp = pd.DataFrame(pair_rows); pp.to_csv(OUT / 'paired_pooled_vs_separate.csv', index=False)
print('Pooled vs separate, paired over all coin-events (exact McNemar; exploratory):'); display(pp)

allev = pd.concat([coin[s]['EVENTS'].assign(coin=s) for s in COINS], ignore_index=True)
bt = allev.breach_utc.astype('int64').to_numpy(); cs = allev.coin.to_numpy()
co = [bool(((np.abs(bt - bt[i]) <= LEAD_MAX * NS) & (cs != cs[i])).any()) for i in range(len(allev))]
print(f'Share of evaluation events with another coin breaching within ±4h: {np.mean(co):.1%} of {len(allev)} events')
pd.DataFrame({'coin': cs, 'breach_utc': allev.breach_utc, 'other_coin_within_4h': co}).to_csv(OUT / 'cross_coin_cooccurrence.csv', index=False)
for name, (thr, al) in results.items():
    share = {s: len(al[s]) for s in COINS}
    print(name, 'alerts by coin:', share)

policy,S0_separate,S1_pooled,S1b_pooled_coincap
coin,,,
AVAXUSDT,2,3,3
BTCUSDT,22,18,18
DOGEUSDT,16,17,17
ETHUSDT,9,15,15
SOLUSDT,12,13,13


,events,alarms,caught,before_1pct
policy,,,,
S0_separate,397,3992,164,61
S1_pooled,397,3999,195,66
S1b_pooled_coincap,397,3999,195,66


Pooled vs separate, paired over all coin-events (exact McNemar; exploratory):


,policy,separate_only_pre1pct,pooled_only_pre1pct,p_pre1pct_two_sided,separate_only_caught,pooled_only_caught,p_caught_two_sided
0,S1_pooled,17,22,0.522397,19,50,0.000244
1,S1b_pooled_coincap,17,22,0.522397,19,50,0.000244


Share of evaluation events with another coin breaching within ±4h: 73.0% of 397 events
S1_pooled alerts by coin: {'BTCUSDT': 740, 'ETHUSDT': 953, 'SOLUSDT': 730, 'DOGEUSDT': 638, 'AVAXUSDT': 938}
S1b_pooled_coincap alerts by coin: {'BTCUSDT': 740, 'ETHUSDT': 953, 'SOLUSDT': 730, 'DOGEUSDT': 638, 'AVAXUSDT': 938}
